In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import base64
import json
import logging
import mimetypes
import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain_openai import OpenAIEmbeddings

load_dotenv()
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")
logger = logging.getLogger("agriagent")

GROQ_API_KEY = os.getenv("GROQ_API_KEY") or os.getenv("groq_api_key")
GROQ_MODEL = os.getenv("GROQ_MODEL", "qwen/qwen3.8-27b")
CHROMA_PERSIST_DIRECTORY = os.getenv("CHROMA_PERSIST_DIRECTORY", "./chroma_db")
CHROMA_COLLECTION_NAME = os.getenv("CHROMA_COLLECTION_NAME", "agri_rag_v2")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "text-embedding-3-small")

if not GROQ_API_KEY:
    raise RuntimeError("GROQ_API_KEY is missing from the environment.")

print(f"GROQ_MODEL={GROQ_MODEL}")
print(f"Chroma collection={CHROMA_COLLECTION_NAME}")
print(f"Chroma directory={CHROMA_PERSIST_DIRECTORY}")

GROQ_MODEL=qwen/qwen3.8-27b
Chroma collection=agri_rag_v2
Chroma directory=./chroma_db


In [3]:
def _metadata_value(metadata: dict[str, Any], *keys: str) -> str:
    for key in keys:
        value = metadata.get(key)
        if value not in (None, ""):
            return str(value)
    return "unknown"


embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

# Open the existing collection. This does not ingest, re-embed, or create a collection.
vectorstore = Chroma(
    collection_name=CHROMA_COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=CHROMA_PERSIST_DIRECTORY,
)

retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 6,
        "fetch_k": 30,
        "lambda_mult": 0.7,
    },
)

print(f"Existing Chroma records: {vectorstore._collection.count()}")


@tool
def retrieve_plant_knowledge(query: str) -> str:
    """Retrieve source-aware plant disease knowledge for a natural-language query."""
    if not query or not query.strip():
        return json.dumps({"query": query, "results": [], "error": "The query is empty."})

    logger.info("retrieve_plant_knowledge query=%s", query)
    documents = retriever.invoke(query.strip())
    results = []

    for document in documents:
        metadata = dict(document.metadata or {})
        results.append(
            {
                "text": document.page_content,
                "source": _metadata_value(metadata, "source", "url", "file_path"),
                "title": _metadata_value(metadata, "title", "document_title"),
                "source_type": _metadata_value(metadata, "source_type"),
                "page": metadata.get("page", metadata.get("page_number")),
                "metadata": metadata,
            }
        )

    logger.info("retrieved_documents=%d", len(results))
    return json.dumps({"query": query, "results": results}, ensure_ascii=False, default=str)

Existing Chroma records: 827


In [4]:
VISION_SYSTEM_PROMPT = """You inspect a plant image for a retrieval pipeline.
Return only valid JSON with these keys: plant, plant_part, symptoms, lesion_shape,
lesion_color, lesion_pattern, distribution, severity, visible_pests,
visible_fungal_growth, overall_observation, uncertainty.
Describe only what is visibly present. Do not confidently diagnose a disease,
do not invent symptoms, and distinguish observation from interpretation.
Use null or an empty list when a field cannot be determined. Identify the crop
only when visually reasonable and mention uncertainty where appropriate."""


def _extract_json(text: str) -> dict[str, Any]:
    cleaned = text.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1].rsplit("```", 1)[0].strip()
    try:
        value = json.loads(cleaned)
    except json.JSONDecodeError as error:
        raise ValueError(f"Vision model returned invalid JSON: {error}") from error
    if not isinstance(value, dict):
        raise ValueError("Vision model response must be a JSON object.")
    return value


@tool
def analyze_plant_image(image_path: str) -> str:
    """Extract structured visible evidence from a local plant image without diagnosing it."""
    path = Path(image_path).expanduser()
    allowed_types = {".jpg", ".jpeg", ".png", ".webp"}

    if not path.is_file():
        return json.dumps({"error": f"Image not found: {path}"})
    if path.suffix.lower() not in allowed_types:
        return json.dumps({"error": "Supported image types are JPG, JPEG, PNG, and WEBP."})
    if path.stat().st_size > 10 * 1024 * 1024:
        return json.dumps({"error": "Image must be 10 MB or smaller."})

    mime_type = mimetypes.guess_type(path.name)[0] or "image/jpeg"
    image_data = base64.b64encode(path.read_bytes()).decode("ascii")
    vision_llm = ChatGroq(
        model=GROQ_MODEL,
        api_key=GROQ_API_KEY,
        temperature=0,
    )

    message = [
        {"type": "text", "text": VISION_SYSTEM_PROMPT},
        {
            "type": "image_url",
            "image_url": {"url": f"data:{mime_type};base64,{image_data}"},
        },
    ]

    logger.info("analyze_plant_image path=%s", path)
    response = vision_llm.invoke([{"role": "user", "content": message}])
    content = response.content if hasattr(response, "content") else str(response)
    observations = _extract_json(content)
    logger.info("image_analysis_completed")
    return json.dumps(observations, ensure_ascii=False)

In [5]:
FINAL_SYSTEM_PROMPT = """You are a careful plant disease assistant.
Use only the retrieved knowledge as factual support. Treat image observations as
uncertain visual evidence, not as a diagnosis. Do not expose internal tool calls
or reasoning. Do not invent sources or unsupported treatments.

For image questions, answer with these headings:
Visual observations:
Most consistent possibility:
Why:
Other possibilities:
Recommended next steps:
Sources:

For text-only questions, answer clearly and include Sources. If retrieval is
empty or insufficient, say so explicitly."""


def _build_retrieval_query(question: str, observations: dict[str, Any] | None) -> str:
    if not observations:
        return question.strip()

    evidence = [question.strip()]
    for key in (
        "plant",
        "plant_part",
        "lesion_shape",
        "lesion_color",
        "lesion_pattern",
        "distribution",
        "severity",
        "overall_observation",
    ):
        value = observations.get(key)
        if value not in (None, "", [], {}):
            evidence.append(f"{key.replace('_', ' ')}: {value}")
    symptoms = observations.get("symptoms") or []
    if symptoms:
        evidence.append(f"visible symptoms: {', '.join(map(str, symptoms))}")
    return ". ".join(evidence)


def _source_lines(retrieval_payload: dict[str, Any]) -> str:
    lines = []
    for result in retrieval_payload.get("results", []):
        source = result.get("source", "unknown")
        title = result.get("title", "unknown")
        page = result.get("page")
        page_text = f", page {page}" if page is not None else ""
        lines.append(f"- {title} | {source}{page_text}")
    return "\n".join(dict.fromkeys(lines)) or "- No sources were retrieved."


def run_plant_agent(question: str, image_path: str | None = None) -> dict[str, Any]:
    """Run the two-tool plant agent and return debug data plus a grounded answer."""
    if not question or not question.strip():
        raise ValueError("Please enter a non-empty question.")

    observations = None
    if image_path:
        observations = json.loads(analyze_plant_image.invoke({"image_path": image_path}))
        if observations.get("error"):
            return {"answer": observations["error"], "observations": observations, "retrieval_query": None, "retrieval": {}}

    retrieval_query = _build_retrieval_query(question, observations)
    retrieval = json.loads(retrieve_plant_knowledge.invoke({"query": retrieval_query}))
    results = retrieval.get("results", [])
    context = "\n\n---\n\n".join(result["text"] for result in results)
    observations_text = json.dumps(observations, ensure_ascii=False, indent=2) if observations else "No image provided."
    sources = _source_lines(retrieval)

    final_llm = ChatGroq(
        model=GROQ_MODEL,
        api_key=GROQ_API_KEY,
        temperature=0,
    )
    prompt = f"""{FINAL_SYSTEM_PROMPT}

User question:
{question}

Visual observations:
{observations_text}

Retrieved knowledge:
{context or 'No relevant documents were retrieved.'}

Retrieved sources:
{sources}"""
    logger.info("final_generation")
    response = final_llm.invoke(prompt)
    answer = response.content if hasattr(response, "content") else str(response)

    return {
        "answer": answer,
        "observations": observations,
        "retrieval_query": retrieval_query,
        "retrieval": retrieval,
    }

In [6]:
# Set this to a local JPG, JPEG, PNG, or WEBP file to test image input.
IMAGE_PATH = r""
TEXT_QUESTION = "What are common causes of leaf spots in crops, and what should I check first?"
IMAGE_QUESTION = "What visible symptoms do you observe, and what plant disease possibilities should I investigate?"


def print_agent_result(label: str, result: dict[str, Any]) -> None:
    print(f"\n=== {label} ===")
    print("Answer:")
    print(result["answer"])
    print("\nRetrieval query:")
    print(result.get("retrieval_query"))
    if result.get("observations") is not None:
        print("\nImage observations:")
        print(json.dumps(result["observations"], indent=2, ensure_ascii=False))
    print("\nRetrieved sources:")
    for item in result.get("retrieval", {}).get("results", []):
        print(f"- {item.get('title', 'unknown')} | {item.get('source', 'unknown')}")


text_result = run_plant_agent(TEXT_QUESTION)
print_agent_result("TEXT-ONLY RAG TEST", text_result)

if IMAGE_PATH.strip():
    image_result = run_plant_agent(IMAGE_QUESTION, IMAGE_PATH)
    print_agent_result("IMAGE + TEXT RAG TEST", image_result)
else:
    print("\nImage test skipped: set IMAGE_PATH to a local JPG, JPEG, PNG, or WEBP file and rerun this cell.")

INFO retrieve_plant_knowledge query=What are common causes of leaf spots in crops, and what should I check first?
INFO HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO retrieved_documents=6
INFO final_generation
INFO HTTP Request: POST https://api.groq.com/openai/v1/chat/completions "HTTP/1.1 200 OK"



=== TEXT-ONLY RAG TEST ===
Answer:
Visual observations:
No image provided.

Most consistent possibility:
Leaf spots in crops are commonly caused by fungal, bacterial, or viral pathogens, as well as environmental stressors. Based on the retrieved knowledge, common fungal causes include *Cercospora* spp., *Septoria* spp., *Alternaria* spp., *Botrytis* spp., *Helminthosporium oryzae* (Brown spot in rice), and *Curvularia* (in cumbu). Bacterial causes include *Pseudomonas* species (e.g., *Pseudomonas lachrymans* causing angular leaf spot) and other bacteria causing greasy spots.

Why:
The retrieved sources list specific pathogens associated with leaf spots across various crops:
*   **Fungal:** *Cercospora capsici* (Chillies), *Septoria lycopersici* (Tomato), *Cercospora malayensis* (Bhendi), *Alternaria solani* (Tomato/Brinjal), *Botrytis* sp. (Onion/Watermelon), *Helminthosporium oryzae* (Rice), and *Curvularia* (Cumbu).
*   **Bacterial:** *Pseudomonas lachrymans* (Watermelon) and unspec